# Experiment 6 - Part 1: A Vector Database Knowledge Base with Qdrant

**Course:** HAAI302COM - Artificial Intelligence Laboratory

In Experiment 5 the vectors lived in a NumPy array or a FAISS index inside the notebook's memory. A production knowledge base needs more: it must **persist** data, store **metadata** next to each vector, **filter** by that metadata, accept updates while serving queries, and be shared by many applications. That is what a vector database provides. This experiment uses **Qdrant** (open source, runs locally in Docker) and shows the equivalent **Pinecone** calls (hosted service).

| Part | Topic |
|---|---|
| 1 (this notebook) | collections, points, payloads; ingesting structured records and unstructured PDFs |
| 2 | metadata-filtered search and payload indexes |
| 3 | index configurations: latency and recall benchmark; the Pinecone equivalent |

Our knowledge base has two kinds of content:

- **Structured:** 11,762 arXiv papers sampled across all years (1991 to 2021), each with fields (year, category, number of authors, whether it was published in a journal) plus an abstract.
- **Unstructured:** two research PDFs (the RAG and LoRA papers from Experiment 4) cut into text chunks.

## 0. Connect to Qdrant

Qdrant runs as a server. Start it once from a terminal (this was already done for you in the lab):

```
docker run -d --name lab-qdrant -p 127.0.0.1:6333:6333 qdrant/qdrant
```

`QdrantClient` talks to it over HTTP. Other ways to connect, with the same API: `QdrantClient(":memory:")` (no server, data lost when the notebook ends; fine for Colab but it has no real HNSW index), `QdrantClient(path="local_dir")` (embedded, saved to disk), or `QdrantClient(url=..., api_key=...)` for Qdrant Cloud.

In [1]:
# !pip install -q qdrant-client sentence-transformers pypdf langchain-text-splitters pandas
import os, time, json, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 170)

from qdrant_client import QdrantClient, models
client = QdrantClient(url="http://localhost:6333")
print("collections on the server:", [c.name for c in client.get_collections().collections])

collections on the server: ['arxiv', 'pdf_chunks', 'toy']


## 1. The three building blocks

| Object | What it is |
|---|---|
| **Collection** | A named set of points that share one vector size and one distance metric (like a table). |
| **Point** | One record: an `id` (integer or UUID), a `vector`, and a `payload`. |
| **Payload** | A JSON dictionary of metadata attached to the point (year, source file, the original text...). |

We start with a toy collection of four points in 4 dimensions so that every step can be read by eye.

In [2]:
if client.collection_exists("toy"):
    client.delete_collection("toy")
client.create_collection(
    collection_name="toy",
    vectors_config=models.VectorParams(size=4, distance=models.Distance.COSINE),   # size must match the embeddings; distance fixed at creation
)
print(client.get_collection("toy").status, "| points:", client.count("toy").count)

green | points: 0


**What to notice:** an empty collection with status `green`. Two arguments cannot be changed later without recreating the collection: the vector `size` and the `distance` (`COSINE`, `DOT`, `EUCLID`, `MANHATTAN`). For `COSINE`, Qdrant normalises vectors on insertion.

Now add four points. `upsert` means *insert or replace*: if a point with the same id exists, it is overwritten, which makes loading idempotent.

In [3]:
points = [
    models.PointStruct(id=1, vector=[0.9, 0.1, 0.0, 0.0], payload={"topic": "cats", "year": 2020}),
    models.PointStruct(id=2, vector=[0.8, 0.2, 0.1, 0.0], payload={"topic": "cats", "year": 2023}),
    models.PointStruct(id=3, vector=[0.0, 0.1, 0.9, 0.2], payload={"topic": "cars", "year": 2021}),
    models.PointStruct(id=4, vector=[0.1, 0.0, 0.8, 0.3], payload={"topic": "cars", "year": 2023}),
]
result = client.upsert("toy", points=points)
print(result.status, "| points now:", client.count("toy").count)

completed | points now: 4


In [4]:
# read points back: by id, and by scrolling through the whole collection
print(client.retrieve("toy", ids=[2], with_vectors=True)[0])
records, next_page = client.scroll("toy", limit=10, with_payload=True)
pd.DataFrame([{"id": r.id, **r.payload} for r in records])

id=2 payload={'topic': 'cats', 'year': 2023} vector=[0.9630868, 0.2407717, 0.12038585, 0.0] shard_key=None order_value=None


,id,topic,year
0,1,cats,2020
1,2,cats,2023
2,3,cars,2021
3,4,cars,2023


**What to notice:** `retrieve` fetches known ids; `scroll` pages through a collection without a query (`next_page` is the cursor for the next page, `None` when done). Note that the stored vector of point 2 has length 1 (cosine normalisation happened on insert).

Search with `query_points`. The query is a vector; the result is a list of `ScoredPoint` objects.

In [5]:
hits = client.query_points("toy", query=[1.0, 0.0, 0.0, 0.0], limit=3, with_payload=True).points
pd.DataFrame([{"id": h.id, "score": round(h.score, 3), **h.payload} for h in hits])

,id,score,topic,year
0,1,0.994,cats,2020
1,2,0.963,cats,2023
2,4,0.116,cars,2023


**What to notice:** the two "cats" points come first with cosine scores close to 1, the "cars" point last. For cosine and dot product, a higher score is better (unlike Chroma in Experiment 4, which returned a distance).

## 2. Structured data: records with metadata

Real knowledge bases start from tables. We load the 11,762 sampled arXiv papers and look at what kinds of fields we have, because the field type decides what filters are possible later.

In [6]:
papers = pd.read_json("data/arxiv_sample.jsonl", lines=True)
print(papers.shape)
papers.drop(columns=["abstract"]).head(4)

(11762, 10)


,arxiv_id,title,year,month,categories,primary_category,n_authors,has_journal_ref,n_versions
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and LHC...,2007,4,[hep-ph],hep-ph,4,True,1
1,0704.0171,Discovery of a point-like very-high-energy gamma-ray source in Monoceros,2007,4,[astro-ph],astro-ph,2,False,1
2,0704.0341,Infrared Evolution Equations: Method and Applications,2007,4,[hep-ph],hep-ph,3,True,1
3,0704.0511,A unified approach to SIC-POVMs and MUBs,2007,4,"[quant-ph, math-ph, math.MP]",quant-ph,2,True,1


In [7]:
print(papers.dtypes.to_string())
print("\nyears:", papers.year.min(), "to", papers.year.max(), "| distinct primary categories:", papers.primary_category.nunique())

arxiv_id               str
title                  str
abstract               str
year                 int64
month                int64
categories          object
primary_category       str
n_authors            int64
has_journal_ref       bool
n_versions           int64

years: 1991 to 2021 | distinct primary categories: 166


**What to notice:** each column becomes a payload field, and its type decides which filters work later. Strings (`primary_category`) are exact-match values, integers (`year`, `month`, `n_authors`, `n_versions`) support ranges, the boolean `has_journal_ref` is true or false, and `categories` is a **list**: a paper in `[quant-ph, math-ph, math.MP]` matches a filter on any one of them. The 166 distinct primary categories mean category filters will be very selective. The long `abstract` is stored too, so that a search hit can be displayed without a second lookup.

We also check how the records are distributed, because it matters for filter experiments in Part 2 (a filter that keeps 50 percent of the data behaves very differently from one that keeps 0.1 percent).

In [8]:
top = papers.primary_category.value_counts()
print(top.head(8).to_string())
print("\npapers per decade:", papers.groupby((papers.year // 10) * 10).size().to_dict())

primary_category
hep-ph               709
hep-th               586
astro-ph             571
quant-ph             534
cond-mat.mes-hall    332
gr-qc                321
cs.CV                284
cs.LG                271

papers per decade: {1990: 709, 2000: 2700, 2010: 6236, 2020: 2117}


**What to notice:** the biggest category, hep-ph, holds only 709 papers (6 percent), and the eight largest categories together hold about 31 percent, so a typical category keeps well under 1 percent of the collection (11,762 papers over 166 categories is about 71 per category). Years are skewed to recent decades: 709 papers from the 1990s, 2,700 from the 2000s, 6,236 from the 2010s and 2,117 from 2020 and 2021 only. So `year >= 2010` is a broad filter, while a single category is a very narrow one.

## 3. Embedding the abstracts

We embed `title + abstract` with the same MiniLM model as in Experiment 5, limited to 128 tokens to keep CPU time reasonable. The result is cached in a `.npy` file.

In [9]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.max_seq_length = 128

texts = (papers.title + ". " + papers.abstract).tolist()
if os.path.exists("emb_arxiv.npy"):
    E = np.load("emb_arxiv.npy")
else:
    t0 = time.time()
    E = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False).astype("float32")
    np.save("emb_arxiv.npy", E)
    print(f"encoded {len(texts)} texts in {time.time() - t0:.0f} s")
print(E.shape, E.dtype)

(11762, 384) float32


## 4. Create the collection and ingest

Each paper becomes one point. The payload holds every metadata field *and* the text, so that a search result is self-contained and the application does not need a second lookup. `upload_points` sends the points in batches (`batch_size`) and can use several parallel workers.

Before ingesting we set one option on purpose: `OptimizersConfigDiff(indexing_threshold=...)`. Qdrant only builds the HNSW graph for a segment after it exceeds this size in kilobytes (default 20,000 KB). A small collection below the threshold is searched by brute force. We lower it so that our 11,762 points get a real HNSW index.

In [10]:
NAME = "arxiv"
if client.collection_exists(NAME):
    client.delete_collection(NAME)
client.create_collection(
    NAME,
    vectors_config=models.VectorParams(size=E.shape[1], distance=models.Distance.COSINE),
    optimizers_config=models.OptimizersConfigDiff(indexing_threshold=1000),      # KB; build HNSW once a segment is over 1 MB
)

def payload_of(row):
    return {"arxiv_id": row.arxiv_id, "title": row.title, "abstract": row.abstract, "year": int(row.year), "month": int(row.month),
            "primary_category": row.primary_category, "categories": list(row.categories), "n_authors": int(row.n_authors),
            "has_journal_ref": bool(row.has_journal_ref), "n_versions": int(row.n_versions)}

pts = [models.PointStruct(id=i, vector=E[i].tolist(), payload=payload_of(r)) for i, r in enumerate(papers.itertuples())]
t0 = time.time()
client.upload_points(NAME, pts, batch_size=256, parallel=2, wait=True)
print(f"uploaded {len(pts)} points in {time.time() - t0:.1f} s")

uploaded 11762 points in 4.4 s


In [11]:
def wait_green(name, timeout=120):
    t0 = time.time()
    while client.get_collection(name).status != models.CollectionStatus.GREEN and time.time() - t0 < timeout:
        time.sleep(0.5)
    return client.get_collection(name)

info = wait_green(NAME)
pd.Series({"status": info.status.value, "points_count": info.points_count, "indexed_vectors_count": info.indexed_vectors_count,
           "segments_count": info.segments_count, "vector size": info.config.params.vectors.size,
           "distance": info.config.params.vectors.distance.value}).to_frame("value")

,value
status,green
points_count,11762
indexed_vectors_count,11762
segments_count,8
vector size,384
distance,Cosine


**What to notice:** 11,762 points were uploaded in about 4 seconds. The status is `green`, and `indexed_vectors_count` equals `points_count`, which means every vector is in the HNSW graph because we lowered `indexing_threshold`. (This collection holds about 18 MB of vectors, under the default threshold of 20,000 KB, so with the default it would have stayed unindexed and been searched by brute force.) Note also `segments_count = 8`: Qdrant splits the collection into several segments, each with its own HNSW graph, searches all of them and merges the answers. The server was started with 18 CPU cores, which influences the default number of segments.

The index parameters were chosen by default. They are the knobs of Part 3:

In [12]:
h = info.config.hnsw_config
print("HNSW defaults -> m:", h.m, "| ef_construct:", h.ef_construct, "| full_scan_threshold:", h.full_scan_threshold)
print("indexing_threshold:", info.config.optimizer_config.indexing_threshold)

HNSW defaults -> m: 16 | ef_construct: 100 | full_scan_threshold: 10000
indexing_threshold: 1000


**What to notice:** `m` is the number of graph links per node and `ef_construct` the search width while building (the same ideas as `M` and `efConstruction` in FAISS HNSW from Experiment 5). `full_scan_threshold` says that if a filter leaves fewer than that many candidates (in KB), Qdrant skips the graph and scans them exactly, which we exploit in Part 2.

## 5. Searching the structured collection

A first query. The query text is embedded with the same model (same 128-token limit) and sent as a vector.

In [13]:
def search(text, k=5, **kw):
    q = model.encode(text, normalize_embeddings=True).tolist()
    res = client.query_points(NAME, query=q, limit=k, with_payload=True, **kw).points
    return pd.DataFrame([{"score": round(r.score, 3), "year": r.payload["year"], "cat": r.payload["primary_category"],
                          "title": r.payload["title"][:75]} for r in res])

search("graph neural networks for molecule property prediction")

,score,year,cat,title
0,0.751,2021,cs.LG,Molecule3D: A Benchmark for Predicting 3D Geometries from Molecular Graphs
1,0.680,2021,cs.LG,Geometry-aware Transformer for molecular property prediction
2,0.663,2020,cs.LG,Autoencoding Undirected Molecular Graphs With Neural Networks
3,0.643,2020,physics.chem-ph,OrbNet: Deep Learning for Quantum Chemistry Using Symmetry-Adapted Atomic-O
4,0.630,2021,cs.LG,Calibrated Uncertainty for Molecular Property Prediction using Ensembles of


**What to notice:** the five hits are all about molecules and graph networks (cosine scores 0.63 to 0.75), and four of five are from 2020 or 2021, which is what we expect for a query about graph neural networks. The payload (`year`, `cat`, `title`) came back with each hit, so the application needs no second lookup. Only now that we have payload fields can we ask for the same query restricted to, say, a time range; that is Notebook 02.

## 6. Unstructured data: PDFs into chunks into points

Unstructured documents have no fields, so we create them: load the text, cut it into chunks (as in Experiment 4) and store each chunk as a point whose payload records where it came from (`source`, `page`, `chunk`) and the text itself. The same collection design then supports filters such as "only this PDF" or "only pages after 10".

In [14]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = []
for fname in ["rag_lewis_2020.pdf", "lora_hu_2021.pdf"]:
    for page_no, page in enumerate(PdfReader(f"data/{fname}").pages, start=1):
        text = page.extract_text().replace("ﬁ", "fi").replace("ﬂ", "fl")      # fix ligatures
        for j, piece in enumerate(splitter.split_text(text)):
            chunks.append({"source": fname, "page": page_no, "chunk": j, "text": piece})
chunk_df = pd.DataFrame(chunks)
print(len(chunk_df), "chunks |", chunk_df.groupby("source").size().to_dict())
chunk_df.head(3)

237 chunks | {'lora_hu_2021.pdf': 129, 'rag_lewis_2020.pdf': 108}


,source,page,chunk,text
0,rag_lewis_2020.pdf,1,0,Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick L...
1,rag_lewis_2020.pdf,1,1,"lags behind task-specific architectures. Additionally, providing provenance ..."
2,rag_lewis_2020.pdf,1,2,"pare two RAG formulations, one which conditions on the same retrieved passag..."


In [15]:
model.max_seq_length = 256
C = model.encode(chunk_df.text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False).astype("float32")

PDF = "pdf_chunks"
if client.collection_exists(PDF):
    client.delete_collection(PDF)
client.create_collection(PDF, vectors_config=models.VectorParams(size=C.shape[1], distance=models.Distance.COSINE))
client.upload_points(PDF, [models.PointStruct(id=i, vector=C[i].tolist(), payload=chunk_df.iloc[i].to_dict()) for i in range(len(C))], wait=True)
print("points:", client.count(PDF).count)

points: 237


In [16]:
def pdf_search(text, k=3, flt=None):
    q = model.encode(text, normalize_embeddings=True).tolist()
    res = client.query_points(PDF, query=q, limit=k, query_filter=flt, with_payload=True).points
    return pd.DataFrame([{"score": round(r.score, 3), "source": r.payload["source"], "page": r.payload["page"], "text": r.payload["text"][:90].replace("\n", " ")} for r in res])

pdf_search("Why does LoRA not add inference latency?")

,score,source,page,text
0,0.618,lora_hu_2021.pdf,4,a subset of the pre-trained parameters. LoRA takes a step further and does n...
1,0.578,lora_hu_2021.pdf,5,during training on GPT-3 175B compared to full fine-tuning 5 as we do not ne...
2,0.573,lora_hu_2021.pdf,21,"First of all, LoRA+PE significantly outperforms both LoRA and prefix-embeddi..."


**What to notice:** the three best chunks all come from the LoRA paper (pages 4, 5 and 21) with cosines of 0.57 to 0.62, and the payload tells us the file and the page, which is what a RAG application needs for citations. The same two PDFs were in the Chroma store of Experiment 4; the difference here is that the database runs as a separate service that several programs can query, and it offers filters on `source` and `page` natively.

## 7. Updating and deleting: the database is alive

Unlike a NumPy array, a vector database lets us change data in place. We update one payload field and then delete points that match a condition, using the toy collection so that the real knowledge base stays intact.

In [17]:
print("before:", client.retrieve("toy", ids=[1])[0].payload)
client.set_payload("toy", payload={"reviewed": True}, points=[1])                 # add a new field to one point
print("after :", client.retrieve("toy", ids=[1])[0].payload)

before: {'topic': 'cats', 'year': 2020}
after : {'topic': 'cats', 'year': 2020, 'reviewed': True}


In [18]:
n_before = client.count("toy").count
client.delete("toy", points_selector=models.FilterSelector(filter=models.Filter(must=[
    models.FieldCondition(key="topic", match=models.MatchValue(value="cars"))])))     # delete by condition, not by id
print("points before:", n_before, "| after:", client.count("toy").count, "| topics left:", {r.payload["topic"] for r in client.scroll("toy", limit=10)[0]})

points before: 4 | after: 2 | topics left: {'cats'}


**What to notice:** `set_payload` merged the new field `reviewed` into the existing payload without touching the other fields or the vector, and `delete` with a `FilterSelector` removed both "cars" points (4 points became 2) without our knowing their ids. Because the same filter language is used for search, count, update and delete, learning it once pays off everywhere; Notebook 02 studies it in depth.

## Summary

| Concept | Qdrant call | Seen in this notebook |
|---|---|---|
| Create a table | `create_collection(name, vectors_config=VectorParams(size, distance))` | toy, arxiv, pdf_chunks |
| Insert or replace | `upsert` / `upload_points` | points with vector + payload |
| Read | `retrieve`, `scroll`, `count` | by id and by page |
| Search | `query_points(query=vector, limit=k)` | arxiv and PDF search |
| Update / delete | `set_payload`, `delete(FilterSelector)` | section 7 |
| Inspect | `get_collection` | status, indexed vectors, HNSW defaults |

**Next:** Notebook 02 uses the payload fields for filtered search, adds payload indexes, and measures when filters get slow or lose results.